In [ ]:
!pip install -q -U transformers accelerate bitsandbytes sentencepiece tiktoken
!pip install -q -U peft chromadb langchain langchain-community langchain-core
!pip install -q -U langchain-huggingface langchain-text-splitters

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [2]:
embedding_model = HuggingFaceEmbeddings(
    model_name="Abeshith/research-embedding-finetuned",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True},
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/283 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/575 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/364 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/312 [00:00<?, ?B/s]

In [3]:
biomed_docs = [
    "Hypothalamic glutamate neurotransmission is required for proper energy balance regulation through ARC neurons. EAAC1 knockout mice exhibit hyperphagia and obesity.",
    "Remdesivir clinical trials demonstrated reduced recovery time (10 vs 15 days) in hospitalized COVID-19 patients with moderate efficacy outcomes.",
    "daf-16/FOXO gene in C. elegans mediates longevity phenotypes through insulin signaling pathways.",
    "SARS-CoV-2 rapid testing includes antigen tests (15 min) and molecular PCR assays with high sensitivity.",
    "β1/Ketel protein binds microtubules essential for proper chromosome segregation during cell division.",
    "Cell autonomous sex determination occurs in Galliformes through DMRT1 gene dosage effects.",
]

In [4]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=256,
    chunk_overlap=32,
)

docs = splitter.create_documents(biomed_docs)

In [5]:
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embedding_model,
    persist_directory="./biomed_rag_db"
)

In [6]:
base_model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct"
adapter_name = "Abeshith/llama3-biomedical-rag-lora-v2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

In [7]:
tokenizer = AutoTokenizer.from_pretrained(
    adapter_name,
    use_fast=False,
)

tokenizer_config.json:   0%|          | 0.00/325 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [8]:
tokenizer.pad_token = tokenizer.eos_token

In [9]:
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

In [10]:
model = PeftModel.from_pretrained(base_model, adapter_name)
model.eval()
model.generation_config.max_length = None

adapter_config.json:   0%|          | 0.00/990 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/13.6M [00:00<?, ?B/s]

In [18]:
def retrieve_best_chunks(question, top_k=1):
    docs_with_scores = vectorstore.similarity_search_with_score(question, k=3)

    # lower distance = better in Chroma
    docs_with_scores = sorted(docs_with_scores, key=lambda x: x[1])

    retrieved = []
    for doc, score in docs_with_scores[:top_k]:
        retrieved.append(doc.page_content)

    return retrieved

In [22]:
import re
def generate_rag_answer(question):
    retrieved_chunks = retrieve_best_chunks(question, top_k=1)

    evidence = " ".join(
        [f"[{i+1}] {chunk}" for i, chunk in enumerate(retrieved_chunks)]
    )

    # Add one fixed example to stabilize generation
    prompt = f"""Question: Does remdesivir improve COVID-19 recovery?

Evidence:
[1] Remdesivir clinical trials demonstrated reduced recovery time (10 vs 15 days) in hospitalized COVID-19 patients.

Answer: Remdesivir moderately improves COVID-19 recovery, reducing recovery time from 15 to 10 days.

Question: {question}

Evidence:
{evidence}

Answer:"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=48,
            do_sample=False,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    answer = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True,
    ).strip()

    # cleanup
    answer = answer.split("Question:")[0].strip()
    answer = answer.replace("Answer:", "").strip()

    # keep only first sentence
    sentences = re.split(r'(?<!\\b[A-Z])\\.\\s+', answer)
    answer = ". ".join(sentences[:2]).strip()

    if not answer.endswith("."):
        answer += "."

    return {
        "question": question,
        "retrieved_chunks": retrieved_chunks,
        "answer": answer,
    }

In [23]:
result = generate_rag_answer(
    "Does glutamate regulate energy balance?"
)

print("QUESTION:")
print(result["question"])

print("\nRETRIEVED CHUNKS:")
for chunk in result["retrieved_chunks"]:
    print("-", chunk)

print("\nANSWER:")
print(result["answer"])

QUESTION:
Does glutamate regulate energy balance?

RETRIEVED CHUNKS:
- Hypothalamic glutamate neurotransmission is required for proper energy balance regulation through ARC neurons. EAAC1 knockout mice exhibit hyperphagia and obesity.

ANSWER:
Glutamate is a key neurotransmitter involved in the regulation of energy homeostasis. The hypothalamus is a critical brain region that integrates signals related to nutrient availability and energy status to control food intake and energy expenditure. Here we show.


In [24]:
queries = [
    "Does glutamate regulate energy balance?",
    "How effective is remdesivir for COVID-19?",
    "What is the function of daf-16?",
    "What are the SARS-CoV-2 rapid testing types?",
    "What is the role of β1/Ketel protein?"
]

for q in queries:
    result = generate_rag_answer(q)

    print("=" * 80)
    print("Q:", result["question"])
    print("A:", result["answer"])

Q: Does glutamate regulate energy balance?
A: Glutamate is a key neurotransmitter involved in the regulation of energy homeostasis. The hypothalamus is a critical brain region that integrates signals related to nutrient availability and energy status to control food intake and energy expenditure. Here we show.
Q: How effective is remdesivir for COVID-19?
A: Remdesivir moderately improves COVID-19 recovery, reducing recovery time from 15 to 10 days. The antiviral drug remdesivir has been shown to be moderately effective against the novel coronavirus SARS-CoV-2.
Q: What is the function of daf-16?
A: DAF-16 is a transcription factor that regulates longevity and metabolism in C. elegans. It acts downstream of the insulin receptor DAF-2 and has been shown to be required for the regulation of many genes involved in longevity. The.
Q: What are the SARS-CoV-2 rapid testing types?
A: Rapid SARS-CoV-2 tests include antigen tests that can be performed in 15 minutes and molecular PCR assays that t